# Medical Insurance Cost Prediction - Data Preparation

This notebook focuses on preparing the data for modeling while maintaining data versioning best practices:

1. Create a copy of raw data in interim folder
2. Check and handle duplications
3. Scale numerical features using StandardScaler
4. Feature encoding:
   - Binary variables (sex, smoker): One-hot with drop_first=True
   - Multi-category variables (region, children): Full one-hot encoding
5. Save final processed dataset

## Data Loading and Initial Setup


In [2]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Create necessary directories if they don't exist
Path('../data/interim').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

# Read the original dataset
raw_data_path = Path('../data/raw/insurance.csv')
df = pd.read_csv(raw_data_path)

# Create a copy in the interim folder
interim_data_path = Path('../data/interim/insurance_interim.csv')
df.to_csv(interim_data_path, index=False)

print("Original dataset shape:", df.shape)
print("\nFirst few rows of the original dataset:")
display(df.head())

# Display data types and basic information
print("\nDataset Info:")
print("-" * 50)
print(df.info())


Original dataset shape: (1338, 7)

First few rows of the original dataset:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Dataset Info:
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None


## Check for Duplications

Let's examine if there are any duplicate records in our dataset:
1. Check for exact duplicates (all columns identical)
2. Display duplicate records if any exist
3. Remove duplicates if found


In [3]:
# Check for duplicate rows
duplicate_rows = df.duplicated()
n_duplicates = duplicate_rows.sum()

print(f"Number of duplicate rows: {n_duplicates}")

if n_duplicates > 0:
    print("\nDuplicate records:")
    print("-" * 50)
    display(df[duplicate_rows])
    
    # Remove duplicates and keep the first occurrence
    df = df.drop_duplicates()
    print(f"\nDataset shape after removing duplicates: {df.shape}")
else:
    print("\nNo duplicate records found in the dataset.")


Number of duplicate rows: 1

Duplicate records:
--------------------------------------------------


,age,sex,bmi,children,smoker,region,charges
581,19,male,30.59,0,no,northwest,1639.5631



Dataset shape after removing duplicates: (1337, 7)


## Feature Scaling and Encoding

We'll perform the following transformations:

1. Scale numerical features using StandardScaler:
   - age
   - bmi
   - charges (target variable)

2. Encode categorical features:
   - Binary variables (drop_first=True):
     * sex (female/male)
     * smoker (yes/no)
   - Multi-category variables (full one-hot encoding):
     * region (northeast/northwest/southeast/southwest)
     * children (0/1/2/3/4/5)


In [4]:
# # we devide the dataset into train and test sets in the next step, so we will work with the train set for data preparation. This way, we avoid any potential data leakage from the test set.
# from sklearn.model_selection import train_test_split


# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
# train_df.to_csv(Path('../data/interim/insurance_train.csv'), index=False)
# test_df.to_csv(Path('../data/interim/insurance_test.csv'), index=False)

# numerical_features = ['age', 'bmi'] # except 'charges' which is the target variable


# # Create dummy variables for binary features (with drop_first=True)
# binary_features = ['sex', 'smoker']
# df_binary = pd.get_dummies(train_df[binary_features], drop_first=True)

# # Create dummy variables for region 
# df_region = pd.get_dummies(train_df['region'], prefix='region' , drop_first=True)

# # Create dummy variables for children 
# df_children = pd.get_dummies(train_df['children'], prefix='children' , drop_first=True)

# # Display unique values before encoding
# print("Unique values in 'children' before encoding:")
# print(train_df['children'].value_counts().sort_index())
# print("\nUnique values in 'region' before encoding:")
# print(train_df['region'].value_counts())




# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# train_df[numerical_features] = scaler.fit_transform(train_df[numerical_features])


# # Combine all features
# train_df_processed = pd.concat([
#     train_df[numerical_features], 
#     train_df['charges'],
#     df_binary,  # Binary categorical features (one category dropped)
#     df_region,  
#     df_children 
# ], axis=1)

# # Display the final processed dataset
# print("\nProcessed dataset shape:", train_df_processed.shape)
# print("\nProcessed dataset columns:")
# print("-" * 50)
# print(train_df_processed.columns.tolist())
# print("\nFirst few rows of the processed dataset:")
# display(train_df_processed.head())

# # Save the processed dataset
# processed_data_path_scaled = Path('../data/processed/insurance_train_scaled.csv')
# train_df_processed.to_csv(processed_data_path_scaled, index=False)

# print("\nDataset saved to:", processed_data_path_scaled)


In [5]:
# # Apply the same preprocessing to the test set
# test_df = test_df.copy()

# # Scale numerical features using the scaler fitted on the training set
# test_df[numerical_features] = scaler.transform(test_df[numerical_features])

# # Encode categorical features using the same scheme as training
# test_df_binary = pd.get_dummies(test_df[binary_features], drop_first=True)
# test_df_region = pd.get_dummies(test_df['region'], prefix='region', drop_first=True)
# test_df_children = pd.get_dummies(test_df['children'], prefix='children', drop_first=True)

# test_df_processed = pd.concat([
#     test_df[numerical_features],
#     test_df_binary,
#     test_df_region,
#     test_df_children
# ], axis=1)

# # Ensure the same column layout as the training processed data
# test_df_processed = test_df_processed.reindex(columns=train_df_processed.columns, fill_value=False)

# print("Processed test dataset shape:", test_df_processed.shape)
# print("Processed test dataset columns:")
# print(test_df_processed.columns.tolist())
# display(test_df_processed.head())

# test_processed_path = Path('../data/processed/insurance_test_scaled.csv')
# test_df_processed.to_csv(test_processed_path, index=False)

# print("Processed test dataset saved to:", test_processed_path)

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# we devide the dataset into train and test sets in the next step, so we will work with the train set for data preparation. This way, we avoid any potential data leakage from the test set.
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.to_csv(Path('../data/interim/insurance_train.csv'), index=False)
test_df.to_csv(Path('../data/interim/insurance_test.csv'), index=False)

# Define the preprocessing pipeline
numerical_features = ['age', 'bmi']
binary_features = ['sex', 'smoker']
multi_cat_features = ['region', 'children']


# Split X and y BEFORE preprocessing
X_train_raw = train_df.drop('charges', axis=1)
y_train = train_df['charges']

X_test_raw = test_df.drop('charges', axis=1)
y_test = test_df['charges']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('binary', OneHotEncoder(drop='first', sparse_output=False), binary_features),
        ('multi_cat', OneHotEncoder(sparse_output=False), multi_cat_features)
    ],
    remainder='drop'
)

# Fit the pipeline on train_df and transform
train_df_processed = pd.DataFrame(
    preprocessor.fit_transform(X_train_raw),
    columns=preprocessor.get_feature_names_out() ,
    index=X_train_raw.index
)
train_df_processed['charges'] = y_train.values


# Transform test_df using the fitted pipeline
test_df_processed = pd.DataFrame(
    preprocessor.transform(X_test_raw),
    columns=preprocessor.get_feature_names_out(),
    index=X_test_raw.index
)
test_df_processed['charges'] = y_test.values

# Save the processed datasets
train_df_processed.to_csv(Path('../data/processed/insurance_train_scaled.csv'), index=False)
test_df_processed.to_csv(Path('../data/processed/insurance_test_scaled.csv'), index=False)

# Display the final processed dataset
print("\nProcessed train dataset shape:", train_df_processed.shape)
print("\nProcessed train dataset columns:")
print("-" * 50)
print(train_df_processed.columns.tolist())
print("\nFirst few rows of the processed dataset:")
display(train_df_processed.head())

print("Processed train dataset shape:", train_df_processed.shape)
print("Processed test dataset shape:", test_df_processed.shape)


Processed train dataset shape: (1069, 15)

Processed train dataset columns:
--------------------------------------------------
['num__age', 'num__bmi', 'binary__sex_male', 'binary__smoker_yes', 'multi_cat__region_northeast', 'multi_cat__region_northwest', 'multi_cat__region_southeast', 'multi_cat__region_southwest', 'multi_cat__children_0', 'multi_cat__children_1', 'multi_cat__children_2', 'multi_cat__children_3', 'multi_cat__children_4', 'multi_cat__children_5', 'charges']

First few rows of the processed dataset:


,num__age,num__bmi,binary__sex_male,binary__smoker_yes,multi_cat__region_northeast,multi_cat__region_northwest,multi_cat__region_southeast,multi_cat__region_southwest,multi_cat__children_0,multi_cat__children_1,multi_cat__children_2,multi_cat__children_3,multi_cat__children_4,multi_cat__children_5,charges
1114,-1.157680,-0.996928,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2396.09590
968,-1.300619,-0.792762,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3279.86855
599,0.914926,1.154664,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,33471.97189
170,1.701087,1.806837,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,13405.39030
275,0.557580,-0.651417,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,9715.84100


Processed train dataset shape: (1069, 15)
Processed test dataset shape: (268, 15)


In [7]:
# we do the same for the test set to ensure that the same transformations are applied to both sets. This is crucial for maintaining consistency and ensuring that our model can generalize well to unseen data.





## Summary of Data Preparation Steps

1. **Data Versioning**:
   - Original data preserved in `data/raw/`
   - Working copy created in `data/interim/`
   - Final processed dataset saved in `data/processed/`

2. **Duplication Check**:
   - Checked for exact duplicate records
   - Removed duplicates if found (keeping first occurrence)

3. **Feature Scaling**:
   - Numerical features standardized using StandardScaler:
     * age
     * bmi
     * charges (target variable)
   - All features now have mean = 0 and standard deviation = 1

4. **Feature Encoding**:
   - Binary variables (sex, smoker):
     * One-hot encoded with drop_first=True
     * Reduces multicollinearity
   - Multi-category variables (region, children):
     * Full one-hot encoding (all categories kept)
     * Preserves all category information

5. **Final Dataset**:
   - Duplicates removed (if any)
   - All numerical features standardized
   - Categorical features properly encoded
   - Ready for modeling in the next notebook

The processed dataset maintains all the necessary information while being properly formatted for machine learning algorithms.
